# Clasificación: riesgo de cancelación con Random Forest (Enriquecido)

Este notebook utiliza el dataset `dataset_clasificacion.csv`. Cada fila representa **un pedido completo**. La variable objetivo es `clase_y`: **Cancelado = 1** y **Entregado = 0**.

### Estrategia de Evaluación y Variables Incorporadas:
1. **Demográficas y Cliente**: `edad`, `usuario_id`, `antiguedad_cuenta_dias`, `porcentaje_cancelados_previos`, `dias_desde_ultimo_pedido`.
2. **Financieras y Operativas**: `total`, `costo_envio`, `metodo_pago`, `num_productos`, `total_unidades`, `precio_promedio_unidad`, `unidades_por_producto`, `prop_costo_envio`, `envio_gratis`, `total_real`.
3. **Temporales y Geográficas**: `hora_compra`, `es_fin_de_semana`, `estado_envio`.
4. **Validación Cruzada Estratificada (K-Fold = 5)**: `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`.
5. **Random Forest Regularizado**: `max_depth=12, min_samples_split=10, min_samples_leaf=4, class_weight="balanced"` y `n_estimators=300`.


## 1. Dependencias e importación de librerías


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import (
    make_scorer, classification_report, confusion_matrix, accuracy_score,
    average_precision_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score
)
from sklearn.pipeline import Pipeline

try:
    from IPython.display import display
except ImportError:
    display = print


## 2. Cargar el dataset


In [ ]:
ruta_csv = next((p for p in [
    Path.cwd() / "outputs" / "dataset_clasificacion.csv",
    Path.cwd().parent / "outputs" / "dataset_clasificacion.csv",
    Path.cwd() / "dataset_clasificacion.csv"
] if p.exists()), "outputs/dataset_clasificacion.csv")

df = pd.read_csv(ruta_csv)
print("Total de pedidos en dataset:", len(df))
display(df.head())


## 3. Ingeniería de Características (Feature Engineering Completo)


In [ ]:
X_dict = []
for _, row in df.iterrows():
    total = float(row["total"])
    costo_envio = float(row["costo_envio"])
    num_productos = float(row["num_productos"]) if float(row["num_productos"]) > 0 else 1.0
    total_unidades = float(row["total_unidades"]) if float(row["total_unidades"]) > 0 else 1.0

    fila = {
        "edad": float(row["edad"]) if pd.notnull(row["edad"]) else 0.0,
        "total": total,
        "costo_envio": costo_envio,
        "porcentaje_cancelados_previos": float(row["porcentaje_cancelados_previos"]),
        "metodo_pago": str(row["metodo_pago"]) if pd.notnull(row["metodo_pago"]) else "Sin definir",
        "usuario_id": str(row["usuario_id"]),
        "num_productos": num_productos,
        "total_unidades": total_unidades,
        # Variables Temporales y Geográficas
        "hora_compra": float(row["hora_compra"]) if "hora_compra" in row and pd.notnull(row["hora_compra"]) else 12.0,
        "es_fin_de_semana": float(row["es_fin_de_semana"]) if "es_fin_de_semana" in row and pd.notnull(row["es_fin_de_semana"]) else 0.0,
        "dias_desde_ultimo_pedido": float(row["dias_desde_ultimo_pedido"]) if "dias_desde_ultimo_pedido" in row and pd.notnull(row["dias_desde_ultimo_pedido"]) else 999.0,
        "antiguedad_cuenta_dias": float(row["antiguedad_cuenta_dias"]) if "antiguedad_cuenta_dias" in row and pd.notnull(row["antiguedad_cuenta_dias"]) else 0.0,
        "estado_envio": str(row["estado_envio"]) if "estado_envio" in row and pd.notnull(row["estado_envio"]) else "Hidalgo",
        # Variables Derivadas
        "precio_promedio_unidad": total / total_unidades,
        "unidades_por_producto": total_unidades / num_productos,
        "prop_costo_envio": costo_envio / (total if total > 0 else 1.0),
        "envio_gratis": 1.0 if costo_envio == 0 else 0.0,
        "total_real": total + costo_envio
    }
    X_dict.append(fila)

y = df["clase_y"].astype(int).values
display(df["clase_y"].value_counts().rename({0: "Entregado", 1: "Cancelado"}))


## 4. Validación Cruzada con StratifiedKFold (K = 5)


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipeline = Pipeline([
    ("vectorizacion", DictVectorizer(sparse=True)),
    ("clasificador", RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=4,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

scorers = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "pr_auc": make_scorer(average_precision_score, response_method="predict_proba")
}

cv_results = cross_validate(pipeline, X_dict, y, cv=skf, scoring=scorers)

df_kfold = pd.DataFrame({
    "Fold": [f"Fold {i+1}" for i in range(5)],
    "Accuracy": cv_results["test_accuracy"].round(3),
    "Balanced Accuracy": cv_results["test_balanced_accuracy"].round(3),
    "Precisión": cv_results["test_precision"].round(3),
    "Recall": cv_results["test_recall"].round(3),
    "F1-Score": cv_results["test_f1"].round(3),
    "PR-AUC": cv_results["test_pr_auc"].round(3)
})

print("=== MÉTRICAS POR PLIEGUE (FOLD 1 A 5) ===")
display(df_kfold)

df_promedios = pd.DataFrame({
    "Métrica": ["Accuracy", "Balanced Accuracy", "Precisión", "Recall", "F1-Score", "PR-AUC"],
    "Promedio (Media)": [
        cv_results["test_accuracy"].mean().round(3),
        cv_results["test_balanced_accuracy"].mean().round(3),
        cv_results["test_precision"].mean().round(3),
        cv_results["test_recall"].mean().round(3),
        cv_results["test_f1"].mean().round(3),
        cv_results["test_pr_auc"].mean().round(3)
    ],
    "Desviación Estándar (±)": [
        cv_results["test_accuracy"].std().round(3),
        cv_results["test_balanced_accuracy"].std().round(3),
        cv_results["test_precision"].std().round(3),
        cv_results["test_recall"].std().round(3),
        cv_results["test_f1"].std().round(3),
        cv_results["test_pr_auc"].std().round(3)
    ],
    "Media ± Desv. Estándar": [
        f"{cv_results['test_accuracy'].mean():.3f} ± {cv_results['test_accuracy'].std():.3f}",
        f"{cv_results['test_balanced_accuracy'].mean():.3f} ± {cv_results['test_balanced_accuracy'].std():.3f}",
        f"{cv_results['test_precision'].mean():.3f} ± {cv_results['test_precision'].std():.3f}",
        f"{cv_results['test_recall'].mean():.3f} ± {cv_results['test_recall'].std():.3f}",
        f"{cv_results['test_f1'].mean():.3f} ± {cv_results['test_f1'].std():.3f}",
        f"{cv_results['test_pr_auc'].mean():.3f} ± {cv_results['test_pr_auc'].std():.3f}"
    ]
})

print("\n=== RESUMEN DE PROMEDIOS K-FOLD = 5 ===")
display(df_promedios)


## 5. Evaluación Detallada con Hold-Out (80% Train / 20% Test) y Matriz de Confusión


In [ ]:
X_train_dict, X_test_dict, y_train, y_test = train_test_split(
    X_dict, y, test_size=0.20, random_state=42, stratify=y
)

pipeline.fit(X_train_dict, y_train)
probabilidad = pipeline.predict_proba(X_test_dict)[:, 1]
prediccion = (probabilidad >= 0.50).astype(int)

cm = confusion_matrix(y_test, prediccion)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Entregado", "Cancelado"], yticklabels=["Entregado", "Cancelado"])
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title("Matriz de Confusión - Random Forest (Test Set)")
plt.show()


## 6. Variables con Mayor Importancia


In [ ]:
vectorizador = pipeline.named_steps["vectorizacion"]
bosque = pipeline.named_steps["clasificador"]
importancias = pd.Series(bosque.feature_importances_, index=vectorizador.get_feature_names_out())
display(importancias.sort_values(ascending=False).head(15).to_frame("Importancia"))


## 7. Entrenar Modelo Final y Guardar .joblib


In [ ]:
pipeline.fit(X_dict, y)
joblib.dump(pipeline, "modelo_random_forest_cancelacion.joblib")
print("✅ Modelo final entrenado y guardado exitosamente como modelo_random_forest_cancelacion.joblib")
